# AEI Index Audit — run the whole analysis

Everything runs off `data/processed/panel.csv` (255 rows, 51 states, 5 waves).
Run cells top to bottom. Roughly 4 minutes.

## 1. Setup

In [ ]:
!pip install -q numpy pandas matplotlib scipy

import os, sys, subprocess
from google.colab import userdata
GH_USER, REPO = "vkenned2", "aei-index-audit"
try: token = userdata.get("GH_TOKEN")
except Exception: token = None
if not os.path.exists(REPO):
    url = f"https://{token}@github.com/{GH_USER}/{REPO}.git" if token else f"https://github.com/{GH_USER}/{REPO}.git"
    subprocess.run(["git","clone",url], check=True)
%cd {REPO}
sys.path.insert(0, ".")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aei_audit import indices as ix, aggregation as ag
plt.rcParams.update({"font.size":9,"axes.spines.top":False,"axes.spines.right":False,
                     "axes.grid":True,"grid.alpha":.25,"figure.dpi":140})
SEED=20260822; INK,ACCENT,MUTE="#2b2b2b","#b03a2e","#c9c9c9"

## 2. Verify the maths, then load

In [ ]:
!python -m pytest tests/ -q

p = pd.read_csv("data/processed/panel.csv")
W = sorted(p.wave.unique())
MAPS = [("Census region","census_region"),("Time zone","time_zone"),
        ("BEA region","bea_region"),("Census division","census_division"),
        ("Federal region","federal_region")]
def roll(g,col):
    a=g.groupby(col)[["usage_share","pop_share_fixed"]].sum()
    a["aui"]=a.usage_share/a.pop_share_fixed; return a
print(p.groupby("wave").agg(states=("state","nunique"), platform=("platform","first"),
                            cadence=("cadence","first"), counts=("usage_count","count")))

## 3. Analysis 1 — scale

**Question:** how much reported inequality survives aggregation?

`theil_decompose` splits total inequality exactly into between-group and
within-group. `share_between` is what a reader at that scale still sees.

In [ ]:
rows=[]
for w in W:
    g=p[p.wave==w]
    for nm,col in MAPS:
        d=ix.theil_decompose(g.aui_fixed.values,g[col].values,weights=g.pop_share_fixed.values)
        rows.append({"wave":w,"grouping":f"{nm} (k={d['n_groups']})",**{k:d[k] for k in
                     ("total","between","within","share_between","check_residual")}})
T1=pd.DataFrame(rows)
print("max |residual| =", T1.check_residual.abs().max(), "  <- must be ~1e-16")
T1.pivot(index="grouping",columns="wave",values="share_between").round(3)

## 4. Analysis 2 — zoning

Five real US government maps of the same country. If the answer moves across
them, the statistic is partly a property of the map.

In [ ]:
rows=[]
for w in W:
    g=p[p.wave==w]; r={"wave":w,"State (k=51)":ix.gini(g.aui_fixed.values,g.pop_share_fixed.values)}
    for nm,col in MAPS:
        a=roll(g,col); r[f"{nm} (k={len(a)})"]=ix.gini(a.aui.values,a.pop_share_fixed.values)
    rows.append(r)
T2=pd.DataFrame(rows).set_index("wave")
mc=[c for c in T2.columns if not c.startswith("State")]
T2["spread_%med"]=100*(T2[mc].max(axis=1)-T2[mc].min(axis=1))/T2[mc].median(axis=1)
T2.round(4)

## 5. Zoning null — the headline

Is the Census map special, or just one map among many? Generate random
**contiguous** 9-region partitions and see where the real map falls.

~90 seconds.

In [ ]:
rows=[]
fig,axes=plt.subplots(1,len(W),figsize=(2.6*len(W),3.2),sharey=True)
for ax,w in zip(np.atleast_1d(axes),W):
    g=p[p.wave==w].set_index("state"); g=g[g.index.isin(ag.CONTIGUOUS_48)]
    sim=ag.zoning_monte_carlo(g.aui_fixed,g.usage_share,g.pop_share_fixed,
                              k=9,stat_fn=ix.gini,n_sims=1500,seed=SEED)
    a=roll(g,"census_division"); obs=ix.gini(a.aui.values,a.pop_share_fixed.values)
    pct=100*(sim<obs).mean()
    rows.append({"wave":w,"observed":obs,"null_median":np.median(sim),"percentile":pct})
    ax.hist(sim,bins=45,color=MUTE,edgecolor="none"); ax.axvline(obs,color=ACCENT,lw=1.8)
    ax.set_title(f"{w}\npctile {pct:.0f}",fontsize=8); ax.set_xlabel("Gini, k=9")
np.atleast_1d(axes)[0].set_ylabel("random maps")
plt.show(); T3=pd.DataFrame(rows); T3.round(4)

## 6. Analysis 4 — how many rankings are real?

Counts exist for the three weekly waves, so resample the state count vector
from a multinomial and see which adjacent ranks survive.

In [ ]:
wave="2026-02"
g=p[(p.wave==wave)&p.usage_count.notna()].copy()
rng=np.random.default_rng(SEED)
c=g.usage_count.to_numpy(float); e=(g.pop_share_fixed/100).to_numpy()
B=rng.multinomial(int(c.sum()),c/c.sum(),size=3000).astype(float)
A=(B/B.sum(1,keepdims=True))/e
g["lo"]=np.percentile(A,2.5,axis=0); g["hi"]=np.percentile(A,97.5,axis=0)
g=g.sort_values("aui_fixed",ascending=False).reset_index(drop=True)

o=np.argsort(-(c/e)); d=A[:,o]; n=d.shape[1]
P=np.array([(d[:,i][:,None]>d).mean(0) for i in range(n)])
adj=np.array([P[i,i+1] for i in range(n-1)])
print(f"adjacent ranks resolved: {100*np.mean((adj>.975)|(adj<.025)):.0f}%")

fig,ax=plt.subplots(figsize=(5,8)); y=np.arange(len(g))
ax.hlines(y,g.lo,g.hi,color=MUTE,lw=1.6); ax.scatter(g.aui_fixed,y,s=12,color=INK,zorder=3)
ax.axvline(1,color=ACCENT,ls="--",lw=1); ax.set_yticks(y); ax.set_yticklabels(g.state,fontsize=6.5)
ax.invert_yaxis(); ax.set_xlabel("AI Usage Index (1.0 = proportional)"); plt.show()

## 7. The two data breaks

Left: inequality falls, but the platform universe changes mid-series.
Right: CA and NY swap share in 2025-11 and swap back.

Neither is a result about behaviour. Both must be reported.

In [ ]:
th=[ix.theil(p[p.wave==w].aui_fixed.values,p[p.wave==w].pop_share_fixed.values) for w in W]
sh=p.pivot(index="state",columns="wave",values="usage_share")
fig,(a1,a2)=plt.subplots(1,2,figsize=(9,3.6))
a1.plot(W,th,marker="o",color=INK); a1.axvspan(1.5,2.5,color=ACCENT,alpha=.12)
a1.set_ylabel("Theil"); a1.set_title("platform universe changes here",fontsize=9)
a1.tick_params(axis="x",labelrotation=30)
for s in ["CA","NY","TX","FL"]: a2.plot(W,sh.loc[s],marker="o",ms=4,label=s)
a2.axvspan(0.5,1.5,color="#4a6fa5",alpha=.12); a2.legend(frameon=False,fontsize=7)
a2.set_ylabel("% of national usage"); a2.set_title("CA/NY attribution anomaly",fontsize=9)
a2.tick_params(axis="x",labelrotation=30); plt.show()

## 8. Regenerate everything and push

In [ ]:
!python scripts/run_analysis.py
!git add -A && git commit -q -m "analysis: figures and tables" && git push -q
print("pushed")